In [196]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
import joblib


In [197]:
df=pd.read_csv(r"P:\AI-Powered Document Classification and Intelligent Indexing\data\csv_dataset\extracted_text.csv")
encoder=LabelEncoder()
df['Category_Encoded'] = encoder.fit_transform(df['Category'])
df.head()

,Category,Content,Category_Encoded
0,finance,finance management s discussion and analysis m...,1
1,finance,finance internal audit compliance memo sox 404...,1
2,finance,finance commercial credit underwriting memo se...,1
3,finance,finance capex budget variance report q2 capex ...,1
4,finance,finance investment committee brief lbo project...,1


In [198]:
embedder=SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

In [199]:
contents=df['Content']
averaged_vector=[]
for content in contents:
    vectors_=[]
    sum=0
    sentences=content.split(" ")
    chunks=[]
    for words in sentences:

        if len(chunks)==200:
            chunk_string = " ".join(chunks)
            vector = embedder.encode(chunk_string)
            vectors_.append(vector)
            chunks=[]
        chunks.append(words)
    if len(chunks) > 0:
        chunk_string = " ".join(chunks)
        vector = embedder.encode(chunk_string)
        vectors_.append(vector)
    if len(vectors_)>0:
        for vector in vectors_:
            sum+=vector
        average=sum/len(vectors_)
        averaged_vector.append(average)
print(averaged_vector)



[array([-2.47214399e-02,  1.04801906e-02, -8.02133605e-03,  2.02645846e-02,
        6.65738136e-02,  3.49148479e-03, -8.97420011e-03,  2.67915633e-02,
       -8.71236771e-02, -2.40482911e-02, -1.07490607e-02,  4.22301404e-02,
        1.47899650e-02,  5.90722859e-02,  2.50384361e-02,  7.09668025e-02,
        2.81149219e-03, -1.68382432e-02, -3.86288799e-02,  1.25872912e-02,
       -4.02770052e-03,  1.66488532e-02, -2.74505410e-02, -1.94686204e-02,
        3.92547203e-03,  2.44212020e-02, -4.14921567e-02, -1.53842038e-02,
       -6.18214952e-04, -6.92811981e-02,  9.38023906e-03, -1.41701140e-02,
        2.58960836e-02, -4.05269768e-03,  2.07194535e-06, -4.38653119e-02,
       -4.20163684e-02,  1.54152839e-02, -1.79697275e-02,  3.60096544e-02,
       -8.24119523e-03, -6.68523461e-03,  2.03163214e-02, -1.02589861e-03,
        7.00100674e-04, -3.54152657e-02,  1.31520387e-02,  3.70654017e-02,
       -2.77568474e-02,  3.04356404e-03,  2.11983901e-02, -7.04892725e-03,
       -3.00098807e-02, 

In [200]:
x_train,x_test,y_train,y_test=train_test_split(averaged_vector,df['Category_Encoded'],test_size=0.3,random_state=42)

In [201]:
model=LogisticRegression(C=1.0,class_weight='balanced', max_iter=1000, random_state=42)
model.fit(x_train,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [202]:
y_pred=model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      1.00      0.93        14
           1       0.85      0.85      0.85        13
           2       0.78      1.00      0.88         7
           3       1.00      0.60      0.75        10
           4       0.83      0.83      0.83         6

    accuracy                           0.86        50
   macro avg       0.87      0.86      0.85        50
weighted avg       0.87      0.86      0.85        50



In [203]:
new_text = ["The company reported a net revenue increase in Q3."]
new_emb = embedder.encode(new_text)

predicted_category = encoder.inverse_transform(model.predict(new_emb))[0]
print(f"Predicted: {predicted_category.upper()}")

Predicted: FINANCE


In [204]:
joblib.dump(model, '../models/Logistic_Model.joblib')
joblib.dump(encoder, '../models/Logistic_Encoder.joblib')
embedder.save('sentence_transformer_model')